# ACTIVE-TB-X v8 - Full 156-Task Track-B Benchmark

This notebook runs the frozen, model-free scenario-synthesis configuration on all 156 VerilogEval Human tasks:

1. five deterministic Random trials at an exact 32-cycle budget,
2. `active_portfolio_no_llm` at the same 32-cycle budget,
3. family-balanced visible/hidden mutation banks and formal equivalence triage,
4. temporal objectives, heuristic exploration, BMC witnesses, and real Verilator coverage,
5. method-level checkpointing and paired task-level confidence intervals.

This is the **RTL-available Track B** protocol: generation can inspect the golden RTL and visible mutants, while hidden mutants are used only for final scoring. It must not be presented as a specification-only AutoBench/CorrectBench comparison.


In [ ]:
# 1) GPU + EDA tools. Do not upgrade Kaggle's base Python stack.
import os, shutil, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

subprocess.run(["nvidia-smi", "-L"], check=True)
missing = [x for x in ["iverilog", "verilator", "verilator_coverage", "yosys", "yosys-smtbmc", "z3"] if shutil.which(x) is None]
if missing:
    print("Installing EDA tools because these are missing:", missing)
    subprocess.run("apt-get update -qq >/tmp/active_tb_apt_update.log 2>&1", shell=True, check=True)
    subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq iverilog verilator yosys z3 >/tmp/active_tb_apt_install.log 2>&1", shell=True, check=True)
for tool in ["iverilog", "vvp", "verilator", "verilator_coverage", "yosys", "yosys-smtbmc", "z3"]:
    path = shutil.which(tool)
    print(f"{tool}: {path}")
    if path is None:
        raise RuntimeError(f"Required EDA tool missing: {tool}")
print("EDA_PREFLIGHT_PASS")


In [ ]:
# 2) Locate/copy this project to writable /kaggle/working.
from pathlib import Path
import shutil, os, zipfile

WORK_PROJECT = Path("/kaggle/working/active_tb_plus_project")
candidates = []
for base in [Path.cwd(), Path("/kaggle/input")]:
    if not base.exists():
        continue
    for p in base.rglob("run_budget_sweep.py"):
        if (p.parent / "active_tb" / "engine.py").exists():
            candidates.append(p.parent)
if not candidates:
    # Kaggle datasets often carry the Windows-built project as one portable
    # archive. Extract only after checking every member stays below the target.
    extract_root = Path("/kaggle/working/active_tb_plus_attached_source")
    archives = sorted(
        p for p in Path("/kaggle/input").rglob("*.zip")
        if "active_tb" in p.name.lower() or "active-tb" in p.name.lower()
    )
    for archive in archives:
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True)
        try:
            with zipfile.ZipFile(archive) as zf:
                for member in zf.infolist():
                    target = (extract_root / member.filename).resolve()
                    if target != extract_root.resolve() and extract_root.resolve() not in target.parents:
                        raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
                zf.extractall(extract_root)
            for p in extract_root.rglob("run_budget_sweep.py"):
                if (p.parent / "active_tb" / "engine.py").exists():
                    candidates.append(p.parent)
        except zipfile.BadZipFile:
            continue
        if candidates:
            print("Extracted project archive:", archive)
            break
if not candidates:
    raise RuntimeError("Could not find ACTIVE-TB+ project directory or portable ZIP in Kaggle inputs.")
source = sorted(candidates, key=lambda p: (0 if str(p).startswith('/kaggle/input') else 1, len(str(p))))[0]
print("Project source:", source)
if source.resolve() != WORK_PROJECT.resolve():
    if WORK_PROJECT.exists():
        shutil.rmtree(WORK_PROJECT)
    shutil.copytree(source, WORK_PROJECT, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "outputs"))
os.chdir(WORK_PROJECT)
print("Working project:", Path.cwd())
assert (WORK_PROJECT / "run_budget_sweep.py").exists()


In [ ]:
# 3) Build the isolated ACTIVE-TB environment without ensurepip.
import subprocess, sys
from pathlib import Path
VENV = Path("/kaggle/working/active_tb_venv")
VENV_PY = VENV / "bin" / "python"
cmd = [
    sys.executable, str(WORK_PROJECT / "bootstrap_kaggle.py"),
    "--venv", str(VENV),
    "--requirements", str(WORK_PROJECT / "requirements.txt"),
]
print("Bootstrap interpreter:", sys.executable)
subprocess.run(cmd, check=True, cwd=WORK_PROJECT)
assert VENV_PY.exists()
print("VENV_PY =", VENV_PY)


In [ ]:
# 4) Dependency and dual-T4 runtime preflight. No model is loaded.
import subprocess
subprocess.run([str(VENV_PY), str(WORK_PROJECT / "environment_smoke_test.py")], check=True, cwd=WORK_PROJECT)
subprocess.run([str(VENV_PY), str(WORK_PROJECT / "check_kaggle_runtime.py")], check=True, cwd=WORK_PROJECT)


In [ ]:
# 5) Frozen v8 configuration is model-free.
QWEN_MODEL_PATH = None
print("Contract enrichment: mechanical parser")
print("Scenario generation: temporal portfolio + heuristic exploration + BMC")
print("LLM model loaded: no")


In [ ]:
# 6) No model-placement preflight is needed.
RUN_FULL_MODEL_LOAD_PREFLIGHT = False
assert QWEN_MODEL_PATH is None


In [ ]:
# 7) Static + checkpoint + adaptive + budget-report smoke tests.
import subprocess
for script in [
    "smoke_test.py",
    "checkpoint_smoke_test.py",
    "adaptive_smoke_test.py",
    "portfolio_smoke_test.py",
    "real_coverage_smoke_test.py",
    "mutation_balance_smoke_test.py",
    "temporal_coverage_smoke_test.py",
    "formal_triage_smoke_test.py",
    "formal_integration_smoke_test.py",
    "budget_report_smoke_test.py",
    "budget_method_smoke_test.py",
]:
    subprocess.run([str(VENV_PY), str(WORK_PROJECT / script)], check=True, cwd=WORK_PROJECT)
print("ACTIVE_TB_PLUS_SMOKE_PASS")


In [ ]:
# 8) Optional local VerilogEval dataset discovery.
from pathlib import Path
MANUAL_LOCAL_DATASET = None
if MANUAL_LOCAL_DATASET:
    LOCAL_DATASET = MANUAL_LOCAL_DATASET
else:
    likely=[]
    for ext in ("*.parquet", "*.jsonl", "*.json"):
        for p in Path("/kaggle/input").rglob(ext):
            if "verilog" in str(p).lower() and "eval" in str(p).lower():
                likely.append(p)
    LOCAL_DATASET = str(sorted(likely, key=lambda p: len(str(p)))[0]) if likely else None
print("LOCAL_DATASET =", LOCAL_DATASET)


In [ ]:
# 9) Resume checkpoint from this exact v8 experiment only.
from pathlib import Path
BUDGET_SWEEP_CHECKPOINT = None
OLD_FULL_RUN_CHECKPOINT = None

def checkpoint_candidates(root="/kaggle/input"):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(str(p) for p in root.rglob("*trackb*v8*checkpoint*.zip"))

print("Checkpoint candidates:")
for i, path in enumerate(checkpoint_candidates()):
    print(f"[{i}] {path}")
print("BUDGET_SWEEP_CHECKPOINT =", BUDGET_SWEEP_CHECKPOINT)


In [ ]:
# 10) Fast exact-budget NO-LLM integration smoke.
# Confirms the new method, strict budgets, repeated Random seeds, BMC, and reports.
import subprocess
SMOKE_OUT = "/kaggle/working/active_tb_plus_budget_smoke"
cmd = [
    str(VENV_PY), str(WORK_PROJECT / "run_budget_sweep.py"),
    "--output", SMOKE_OUT,
    "--limit", "3",
    "--budgets", "8", "16",
    "--methods", "seed_only", "random", "greedy_mutation", "bmc_only", "active_no_llm", "active_adaptive", "active_portfolio_no_llm", "active_portfolio",
    "--random-seeds", "0", "1",
    "--bmc-timeout", "10",
    "--session-hours", "0",
    "--snapshot-every-jobs", "10",
    "--bootstrap-resamples", "200",
]
if LOCAL_DATASET:
    cmd += ["--local-dataset", LOCAL_DATASET]
print(" ".join(map(str,cmd)))
subprocess.run(cmd, check=True, cwd=WORK_PROJECT)


In [ ]:
# 11) LLM smoke is disabled for the frozen model-free benchmark.
RUN_LLM_SMOKE = False


In [ ]:
# 12) Full 156-task RTL-available Track-B benchmark.
RUN_FULL_TRACK_B = True
if RUN_FULL_TRACK_B:
    import subprocess
    OUT = "/kaggle/working/active_tb_x_full_156_trackb_v8"
    cmd = [
        str(VENV_PY), str(WORK_PROJECT / "run_budget_sweep.py"),
        "--output", OUT,
        "--task-start", "0", "--task-end", "156",
        "--budgets", "32",
        "--methods", "random", "active_portfolio_no_llm",
        "--random-seeds", "0", "1", "2", "3", "4",
        "--bmc-timeout", "15",
        "--max-bmc-calls", "8",
        "--adaptive-max-rounds", "20",
        "--exploration-pool", "20",
        "--max-exploration-additions", "6",
        "--coverage-weight", "0.75",
        "--functional-coverage-weight", "1.0",
        "--novelty-weight", "0.35",
        "--family-weight", "0.50",
        "--scheduler-exploration-strength", "0.65",
        "--enable-real-coverage",
        "--real-coverage-interval", "2",
        "--real-coverage-timeout", "90",
        "--mutant-triage-timeout", "4",
        "--max-mutants-per-family", "4",
        "--max-mutants-total", "48",
        "--session-hours", "10.25",
        "--stop-before-minutes", "45",
        "--snapshot-every-jobs", "10",
        "--bootstrap-resamples", "5000",
        "--snapshot-path", "/kaggle/working/active_tb_x_full_156_trackb_v8_checkpoint_latest.zip",
    ]
    if LOCAL_DATASET:
        cmd += ["--local-dataset", LOCAL_DATASET]
    if BUDGET_SWEEP_CHECKPOINT:
        cmd += ["--resume-from", BUDGET_SWEEP_CHECKPOINT]
    print(" ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, cwd=WORK_PROJECT)


In [ ]:
# 13) Enforce full-run invariants and print the paper-facing comparison.
from pathlib import Path
import json
import pandas as pd

def validate_run(root, expected_rows, allowed_budgets, expected_unsupported_tasks=()):
    root = Path(root)
    df = pd.read_csv(root / "per_task_results.csv")
    assert len(df) == expected_rows, f"Expected {expected_rows} rows, found {len(df)}"
    assert not df.duplicated(["task_id", "method"]).any(), "Duplicate jobs"
    parsed_budget = pd.to_numeric(df["method"].astype(str).str.extract(r"^b(\d+)__", expand=False), errors="coerce")
    assert parsed_budget.notna().all()
    assert set(parsed_budget.astype(int)).issubset(set(allowed_budgets))
    supported = df["supported"].astype(str).str.lower().eq("true")
    unsupported = df.loc[~supported].copy()
    unsupported_tasks = set(unsupported["task_id"].astype(str))
    assert unsupported_tasks == set(expected_unsupported_tasks), (
        f"Unexpected unsupported tasks: {sorted(unsupported_tasks)}"
    )
    if unsupported_tasks:
        method_count = df["method"].nunique()
        counts = unsupported.groupby("task_id")["method"].nunique()
        assert (counts == method_count).all(), (
            "Unsupported task exclusions must apply symmetrically to every method"
        )
        print("DOCUMENTED_UNSUPPORTED_TASKS")
        print(unsupported[["task_id", "reason"]].drop_duplicates().to_string(index=False))
    assert (pd.to_numeric(df["suite_cycles"]) <= parsed_budget).all(), "Cycle-budget violation"
    for name in ["aggregate_results.csv", "budget_curve_summary.csv", "paired_bootstrap.csv", "run_state.json", "llm_usage.json"]:
        assert (root / name).exists(), f"Missing {name}"
    return df

smoke_df = validate_run("/kaggle/working/active_tb_plus_budget_smoke", 54, {8, 16})
root = Path("/kaggle/working/active_tb_x_full_156_trackb_v8")
full_df = validate_run(
    root,
    936,
    {32},
    expected_unsupported_tasks={"review2015_fancytimer", "review2015_fsm", "conwaylife"},
)
state = json.loads((root / "run_state.json").read_text())
assert state.get("status") == "complete", state
usage = json.loads((root / "llm_usage.json").read_text())
assert usage["contract_enrichment"]["calls"] == 0, usage
assert usage["scenario_generation"]["calls"] == 0, usage
assert not usage["contract_enrichment_enabled"], usage
assert Path("/kaggle/working/active_tb_x_full_156_trackb_v8_checkpoint_latest.zip").exists()

print("\n=== FULL TRACK-B SUMMARY ===")
print(pd.read_csv(root / "budget_curve_summary.csv").to_string(index=False))
print("\n=== PAIRED TASK-LEVEL COMPARISON ===")
print(pd.read_csv(root / "paired_bootstrap.csv").to_string(index=False))
print("\n=== LLM USAGE (MUST BE ZERO) ===")
print(json.dumps(usage, indent=2))

PRESERVE_DISPOSABLE_RUNTIME = False
if not PRESERVE_DISPOSABLE_RUNTIME:
    import os, shutil
    os.chdir("/kaggle/working")
    for disposable in [Path(VENV), Path(WORK_PROJECT), Path("/kaggle/working/active_tb_plus_attached_source")]:
        if disposable.exists():
            shutil.rmtree(disposable)
    print("DISPOSABLE_RUNTIME_CLEANUP_PASS")


## Resume policy

The runner checkpoints each completed `(task, method)` job and writes `active_tb_x_full_156_trackb_v8_checkpoint_latest.zip`. If Kaggle stops early, attach that ZIP to a new session, set `BUDGET_SWEEP_CHECKPOINT`, and rerun. The experiment fingerprint rejects incompatible configurations.
